# 4 · A zoo of finite element spaces 🔬

:::{dropdown} ▶ How to run / view this notebook
:class: howto-run

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=04-fespaces.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/04-fespaces.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>
:::

:::{dropdown} 🎭 Story — sharpening your tools
:class: storytelling

*Units 3–6 are the Beast-handler's toolkit — coefficient functions, finite element
spaces, weak forms and solvers. Learn to pick the right tool and the Beast moves where
you point.*
:::

In unit 3 a **CoefficientFunction** was a function of a *mapped integration point*. Here
we make the **basis (shape) functions** systematic and tie them to the **mesh**. A
**finite element space** (`FESpace`) *is* the collection of those basis functions, and
every finite element function is a **linear combination** of them (cf. unit 3). The
coefficients are held in a **`GridFunction`** — which is *itself* a
**CoefficientFunction** (same input→output: give it a mapped point, get a value). With
`FESpace` and `GridFunction` in hand we can finally *look at* the basis: set one
coefficient to `1`, the rest to `0`, and draw.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
# --- Bring the website's UI into this live notebook: the ⚙ View-options panel,
# the foldable story/how-to/quiz/further-reading categories and the gimmicks
# (rolling logo + winking head). Loads static/custom.css + view-options.js via
# notebooks/data/ngsum_ui.py. A no-op on the static-site build. --------------
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    sys.path.insert(0, os.path.join(os.getcwd(), "data"))
    try:
        import ngsum_ui; ngsum_ui.enable()
    except Exception:
        pass

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.4))

A `GridFunction` lives in a space and **is** a CoefficientFunction — it evaluates at a
mapped point just like the fields of unit 3, so it composes with everything (`grad`,
arithmetic, drawing). Its coefficients pick out one finite element function.

In [ ]:
gf_demo = GridFunction(H1(mesh, order=2))
gf_demo.Set(x * y)                                   # coefficients fix one FE function
print("a GridFunction is a CoefficientFunction:  gf(0.3, 0.4) =",
      round(gf_demo(mesh(0.3, 0.4)), 3))

In [ ]:
def shape_functions(space, dofs):
    """A multidim GridFunction holding one basis function per requested dof."""
    gf = GridFunction(space, multidim=len(dofs))
    for i, d in enumerate(dofs):
        gf.vecs[i][:] = 0
        gf.vecs[i][d] = 1                  # activate a single basis function
    return gf

## 1. `H1` — continuous shape functions

The Lagrange-type `H1` shape functions are **continuous** across element edges:
their graphs join up with no jumps. That is exactly what makes `H1` the right
home for second-order problems like Poisson and heat. Press play to flip through
a few of them.

In [ ]:
fesH1 = H1(mesh, order=3)
gf = shape_functions(fesH1, [10, 18, 25, 33])
Draw(gf, mesh, "H1 shape fn", interpolate_multidim=False, animate=True,
     deformation=True)

## 2. `L2` — discontinuous shape functions

`L2` functions live **independently on each element** — neighbouring elements
share nothing, so a shape function is a bump confined to a single triangle. This
locality is what the discontinuous-Galerkin transport scheme will exploit
(notebook 12), and it gives the block-diagonal mass matrix.

In [ ]:
fesL2 = L2(mesh, order=3)
gf = shape_functions(fesL2, [12, 20, 28, 36])
Draw(gf, mesh, "L2 shape fn", interpolate_multidim=False, animate=True,
     deformation=True)

## 3. `HDiv` and `HCurl` — vector-valued, partially continuous

Not every space is scalar. `HDiv` shape functions are **vector fields** whose
*normal* component is continuous across edges (ideal for fluxes / flow);
`HCurl` keeps the *tangential* component continuous (ideal for electromagnetics).
Drawn as arrows, you can see the field is well-defined across an edge in one
direction but may jump in the other.

In [ ]:
fesHDiv = HDiv(mesh, order=2)
gf = shape_functions(fesHDiv, [15, 30, 45, 55])
Draw(gf, mesh, "HDiv shape fn", interpolate_multidim=False, animate=True)

## 4. The whole catalogue — every space, by value type

Beyond the four above NGSolve exposes **many** more spaces, all sharing the same
`(mesh, order=…)` / `.ndof` / `.TnT()` interface. We can ask Python for **all** of them
and sort them by the **value type** of a field they hold — *scalar*, *vector* or
*matrix* — read off a trial function's `.dim`. Pick any name and drop it into the
`shape_functions` helper above to see what it is made of.

In [ ]:
import ngsolve, os
from collections import defaultdict

def all_fespaces():
    """Every FESpace class reachable from the `ngsolve` namespace."""
    return sorted(n for n in dir(ngsolve)
                  if isinstance(getattr(ngsolve, n), type)
                  and issubclass(getattr(ngsolve, n), ngsolve.FESpace)
                  and getattr(ngsolve, n) is not ngsolve.FESpace)

def value_kind(name):
    try:
        d = getattr(ngsolve, name)(mesh, order=1).TnT()[0].dim   # 1 / 2 / 4 on a 2-D mesh
        return {1: "scalar", 2: "vector", 4: "matrix"}.get(d, f"dim {d}")
    except Exception:
        return "wrapper / needs special args"          # base-space wrappers, surface-only, …

groups = defaultdict(list)
saved = devnull = None                                 # mute a couple of chatty C++ ctors
try:
    saved = os.dup(1); devnull = os.open(os.devnull, os.O_WRONLY); os.dup2(devnull, 1)
except Exception:
    saved = None
try:
    for name in all_fespaces():
        groups[value_kind(name)].append(name)
finally:
    if saved is not None:
        os.dup2(saved, 1); os.close(saved); os.close(devnull)

for kind in ["scalar", "vector", "matrix", "wrapper / needs special args"]:
    print(f"{kind:>30}:  {', '.join(groups[kind])}")

## 5. How dofs are classified

Internally NGSolve labels every dof by how it **couples** between elements —
local (interior), interface (shared on edges/faces) or wirebasket (the coarse
skeleton used by `bddc`). This is the very information the solvers in notebook 6
exploit. We can simply ask the space.

In [ ]:
from collections import Counter
fes = H1(mesh, order=3)
kinds = Counter(str(fes.CouplingType(i)).split(".")[-1] for i in range(fes.ndof))
print(f"H1 order 3 — {fes.ndof} dofs:")
for kind, count in kinds.items():
    print(f"  {kind:16s}: {count}")

## 6. Supplementary — wrapper spaces

Some spaces are not built from scratch but **wrap** an existing one, modifying it:

* **`MatrixValued(V)`** — stacks several copies of a scalar space into a
  **matrix-valued** field (handy for tensor unknowns like stress);
* **`Periodic(V)`** — identifies opposite boundaries, so the field is **periodic**;
* **`Compress(V)`** — drops unused dofs (e.g. after marking some inactive);
* **`Discontinuous(V)`** — breaks inter-element continuity, making the space
  **element-local** (the same idea that turns a continuous space into a DG one).

They take a base space and hand back a new `FESpace` with the familiar interface.

In [ ]:
base = H1(mesh, order=2)
print(f"base   H1            : scalar, ndof {base.ndof}")
print(f"MatrixValued(H1)     : value dim {MatrixValued(base).TnT()[0].dim} (2x2), "
      f"ndof {MatrixValued(base).ndof}")
print(f"Periodic(H1)         : ndof {Periodic(base).ndof}  (identifies opposite boundaries — "
      f"takes effect on a periodically-built mesh)")
print(f"Compress(H1)         : ndof {Compress(base).ndof}")
print(f"Discontinuous(H1)    : ndof {Discontinuous(base).ndof}  (continuity broken)")

:::{dropdown} 📚 Further reading
:class: further-reading

- **H(curl) and H(div) spaces** — i-tutorial
  [2.3 HCurl/HDiv](https://docu.ngsolve.org/latest/i-tutorials/unit-2.3-hcurlhdiv/hcurlhdiv.html).
- **The shape-function gallery** lives in NGSolve's docs; every space here is built from
  the same `(mesh, order)` interface.
- **The finite elements behind the spaces** — Schöberl's iFEM:
  [the finite element method](https://jschoeberl.github.io/iFEM/FEM/finiteelements.html), and the
  spaces [H(div)](https://jschoeberl.github.io/iFEM/secondorder/hdiv.html) and
  [H(curl)](https://jschoeberl.github.io/iFEM/Maxwell/HCurl.html).
:::

:::{dropdown} 🧠 Quiz — what is the difference between `L2` and `Discontinuous(H1)`?
:class: quiz
Both are **discontinuous** and span the **same** approximation space — full polynomials
of the given order, independent on each element — so both have an **element-block-diagonal
mass matrix**. The difference is the **basis and construction**:

* **`L2`** is a purpose-built discontinuous space with its *own* shape functions
  (typically hierarchical / $L^2$-orthogonal), the natural choice for DG.
* **`Discontinuous(H1)`** takes the **nodal Lagrange `H1`** basis and **duplicates** the
  shared vertex/edge dofs so nothing couples across elements — a *broken* version of an
  existing conforming space.

Same functions, different dofs: reach for `L2` for DG transport; reach for
`Discontinuous(V)` when you specifically want the broken twin of some space `V`.
:::

Next: **solving** — how the linear system is actually handled
(free dofs, lifting boundary data, static condensation).

In [ ]:
# Navigation to the next unit — shown only in a live notebook (Colab /
# JupyterLite / local Jupyter), never in the rendered website.
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _nb, _title = "05-solving", "5 · Solving — Dirichlet dofs & static condensation"
    if "google.colab" in sys.modules:
        _u = "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
    else:                                           # JupyterLite & local open relative .ipynb links
        _u = _nb + ".ipynb"
    from IPython.display import display, Markdown
    display(Markdown("➡️ **Next unit:** [" + _title + "](" + _u + ")"))